In [1]:
%%bash
set -e

TARBALL=$(curl -fsSL https://nodejs.org/dist/latest-v22.x/SHASUMS256.txt \
          | grep -o 'node-v22\.[0-9]*\.[0-9]*-linux-x64\.tar\.gz' | head -n 1)
curl -fsSL "https://nodejs.org/dist/latest-v22.x/$TARBALL" \
  | tar -xz -C /usr/local --strip-components=1
hash -r
echo "Node $(node -v) at $(which node)"

npm uninstall -g gitnexus >/dev/null 2>&1 || true
npm install -g gitnexus@latest

if [ ! -f /content/GitNexus/gitnexus-web/dist/index.html ]; then
  rm -rf /content/GitNexus
  VER=$(npm view gitnexus version)
  git clone --depth 1 --branch "v$VER" https://github.com/abhigyanpatwari/GitNexus.git /content/GitNexus \
    || git clone --depth 1 https://github.com/abhigyanpatwari/GitNexus.git /content/GitNexus
  cd /content/GitNexus
  npm ci --prefix gitnexus-shared && npm run build --prefix gitnexus-shared
  npm ci --prefix gitnexus-web    && npm run build --prefix gitnexus-web
fi

wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
  -O /usr/local/bin/cloudflared
chmod +x /usr/local/bin/cloudflared

echo "INSTALL DONE"

Node v22.23.5 at /usr/local/bin/node

added 257 packages in 52s

69 packages are looking for funding
  run `npm fund` for details

added 2 packages, and audited 3 packages in 1s

found 0 vulnerabilities

> gitnexus-shared@1.0.0 build
> tsc


added 580 packages, and audited 582 packages in 21s

151 packages are looking for funding
  run `npm fund` for details

18 vulnerabilities (2 low, 8 moderate, 7 high, 1 critical)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.

> gitnexus-web@0.0.0 build
> tsc -b && vite build

vite v8.1.5 building client environment for production...
transforming...✓ 3771 modules transformed.
rendering chunks...
computing gzip size...
dist/index.html                                              2.57 kB │ gzip:   0.79 kB
dist/assets/index-C2_aej58.css                              88.84 kB │ gzip:  13.35 kB
dist/assets/__vite-browser

Cloning into '/content/GitNexus'...
Note: switching to 'c4ecf398de637cf779ab2238e69958cfd5d601ce'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

npm warn deprecated @types/dompurify@3.2.0: This is a stub types definition. dompurify provides its own type definitions, so you do not need this installed.
[plugin rolldown:vite-resolve] Module "node:path" has been externalized for browser compatibility, imported by "/content/GitNexus/gitnexus-web/node_modules/@anthropic-ai/sdk/lib/credentials/types.mjs". See https://vite.dev/guid

In [2]:
import os, subprocess, time, urllib.request

OLLAMA_MODEL = "qwen2.5:7b"

!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

subprocess.run(["pkill", "-f", "ollama serve"])
ollama_env = dict(os.environ,
                  OLLAMA_HOST="127.0.0.1:11434",
                  OLLAMA_ORIGINS="*",
                  OLLAMA_CONTEXT_LENGTH="16384")
ollama = subprocess.Popen(["ollama", "serve"], env=ollama_env,
                          stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=3)
        break
    except Exception:
        time.sleep(1)

!ollama pull {OLLAMA_MODEL}
!nvidia-smi --query-gpu=name,memory.total --format=csv

Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [3]:
import base64, subprocess, urllib.parse
from getpass import getpass

org     = ""
project = ""
repo    = ""
pat     = ""
REPO = "/content/target"

q = lambda s: urllib.parse.quote(s, safe="")
url_git = f"https://dev.azure.com/{q(org)}/{q(project)}/_git/{q(repo)}"
auth = base64.b64encode(f":{pat}".encode()).decode()

for pat_ in ("gitnexus", "docker-server.mjs", "cloudflared tunnel"):
    subprocess.run(["pkill", "-f", pat_])
time.sleep(2)
subprocess.run(["rm", "-rf", REPO])

r = subprocess.run(
    ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}",
     "clone", "--depth", "1", url_git, REPO],
    capture_output=True, text=True)
print(r.stderr.replace(auth, "***").replace(pat, "***"))
assert r.returncode == 0, "CLONE FAILED - see the message above"

!cd {REPO} && gitnexus analyze --index-only --name AIAudit
!gitnexus list

print("\n--- total nodes ---")
!cd {REPO} && gitnexus cypher "MATCH (n) RETURN count(n) AS nodes"
print("\n--- total edges ---")
!cd {REPO} && gitnexus cypher "MATCH ()-[r:CodeRelation]->() RETURN count(r) AS edges"
print("\n--- edges by type ---")
!cd {REPO} && gitnexus cypher "MATCH ()-[r:CodeRelation]->() RETURN r.type AS type, count(r) AS edges ORDER BY edges DESC"

Cloning into '/content/target'...
Updating files:  29% (157/534)
Updating files:  30% (161/534)
Updating files:  31% (166/534)
Updating files:  32% (171/534)
Updating files:  33% (177/534)
Updating files:  34% (182/534)
Updating files:  35% (187/534)
Updating files:  36% (193/534)
Updating files:  37% (198/534)
Updating files:  38% (203/534)
Updating files:  39% (209/534)
Updating files:  40% (214/534)
Updating files:  41% (219/534)
Updating files:  42% (225/534)
Updating files:  43% (230/534)
Updating files:  44% (235/534)
Updating files:  45% (241/534)
Updating files:  46% (246/534)
Updating files:  47% (251/534)
Updating files:  48% (257/534)
Updating files:  49% (262/534)
Updating files:  50% (267/534)
Updating files:  51% (273/534)
Updating files:  52% (278/534)
Updating files:  53% (284/534)
Updating files:  54% (289/534)
Updating files:  55% (294/534)
Updating files:  56% (300/534)
Updating files:  57% (305/534)
Updating files:  58% (310/534)
Updating files:  59% (316/534)
Updat

In [4]:
import os, shutil, subprocess

env = dict(os.environ, GITNEXUS_API_KEY="ollama", OPENAI_API_KEY="ollama")
p = subprocess.Popen(
    ["gitnexus", "wiki",
     "--base-url", "http://localhost:11434/v1",
     "--model", OLLAMA_MODEL,
     "--concurrency", "1"],
    cwd="/content/target", env=env,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()

wiki_dir = "/content/target/.gitnexus/wiki"
if p.returncode == 0 and os.path.isdir(wiki_dir):
    shutil.make_archive("/content/wiki", "zip", wiki_dir)
    from google.colab import files
    files.download("/content/wiki.zip")
else:
    print("Wiki generation failed - see the output above")


  GitNexus Wiki (1.6.12)

  GitNexus Wiki Generator

  Config saved to ~/.gitnexus/config.json


  Wiki generated successfully (1748.8s)

  Mode: full
  Pages: 25
  Output: /content/target/.gitnexus/wiki
  Viewer: /content/target/.gitnexus/wiki/index.html



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import os, re, secrets, subprocess, time, urllib.request, urllib.error

for pat_ in ("gitnexus serve", "docker-server.mjs", "cloudflared tunnel"):
    subprocess.run(["pkill", "-f", pat_])
time.sleep(2)

TOKEN = secrets.token_urlsafe(32)
BACKEND = "127.0.0.1:4747"

def start(cmd, log, **kw):
    return subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT, **kw)

def status(url, token=None):
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    try:
        return urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=10).status
    except urllib.error.HTTPError as e:
        return e.code
    except Exception as e:
        return f"FAILED ({e})"

def fail(msg, log):
    print(open(log).read()[-3000:])
    raise SystemExit(msg)

def open_tunnel(target, log, extra=()):
    proc = start(["cloudflared", "tunnel", "--no-autoupdate", "--url", target, *extra], log)
    for _ in range(60):
        time.sleep(1)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log).read())
        if m:
            return proc, m.group(0)
    fail("No tunnel URL - see the log above", log)

backend_env = dict(os.environ, NODE_OPTIONS="--dns-result-order=ipv4first")
backend = start(["gitnexus", "serve", "--host", "127.0.0.1"], "/content/serve.log", env=backend_env)
for _ in range(90):
    if isinstance(status(f"http://{BACKEND}/api/health"), int):
        break
    time.sleep(1)
else:
    fail("Backend did not answer on 127.0.0.1 - see serve.log above", "/content/serve.log")

tunnel, url = open_tunnel("http://127.0.0.1:4173", "/content/tunnel.log")

env = dict(os.environ,
           PORT="4173",
           GITNEXUS_UPSTREAM_URL=BACKEND,
           GITNEXUS_SERVE_AUTH_TOKEN=TOKEN,
           GITNEXUS_BACKEND_URL=url,
           GITNEXUS_PROXY_TIMEOUT_MS="0")
web = start(["node", "../docker-server.mjs"], "/content/web.log",
            cwd="/content/GitNexus/gitnexus-web", env=env)
for _ in range(30):
    if isinstance(status("http://127.0.0.1:4173/"), int):
        break
    time.sleep(1)
else:
    fail("Web server did not start - see web.log above", "/content/web.log")

ollama_url = None
if status("http://127.0.0.1:11434/api/version") == 200:
    ollama_tunnel, ollama_url = open_tunnel(
        "http://127.0.0.1:11434", "/content/ollama_tunnel.log",
        extra=("--http-host-header", "localhost:11434"))

proxied = status("http://127.0.0.1:4173/api/health", TOKEN)
print("backend direct   :", status(f"http://{BACKEND}/api/health"), "(want 200)")
print("proxy, no token  :", status("http://127.0.0.1:4173/api/health"), "(want 401)")
print("proxy + token    :", proxied, "(want 200)")

if proxied != 200:
    print("\n--- web.log ---");   print(open("/content/web.log").read()[-2000:])
    print("--- serve.log ---");   print(open("/content/serve.log").read()[-2000:])
else:
    print("\nOpen       :", url)
    print("Token      :", TOKEN)
    print("Ollama URL :", ollama_url or "Ollama is not running (run Cell 2)")
    print("Model      :", globals().get("OLLAMA_MODEL", "-"))

backend direct   : 200 (want 200)
proxy, no token  : 401 (want 401)
proxy + token    : 200 (want 200)

Open       : https://looks-red-glucose-collections.trycloudflare.com
Token      : DsNg6HVEMEsW-n3SFLvoi_ufRTJUuXo-uZTFBVzwwc8
Ollama URL : https://registrar-thriller-grill-examining.trycloudflare.com
Model      : qwen2.5:7b


In [7]:
!tail -n 30 /content/serve.log /content/web.log /content/tunnel.log /content/ollama.log

# To stop everything, uncomment:
# for pat_ in ("gitnexus serve", "docker-server.mjs", "cloudflared tunnel", "ollama serve"):
#     subprocess.run(["pkill", "-f", pat_])

==> /content/serve.log <==

  GitNexus Serve (1.6.12)
{"level":30,"time":1794490775134,"name":"gitnexus","msg":"MCP HTTP endpoints mounted at /api/mcp"}
GitNexus server running on http://127.0.0.1:4747

==> /content/web.log <==
gitnexus-web listening on http://0.0.0.0:4173

==> /content/tunnel.log <==
2026-11-12T13:39:39Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-11-12T13:39:39Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026/11/12 13:39:39 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 416 kiB). See https://github.com/quic-go/quic-go/wiki/UDP-Buffer-Sizes for details.
2026-11-12T13:39:39Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
2026-11-12T13:39:39Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
2026-11-12T13:39:39Z INF Starting metrics server on 127.0.0.1:20241/metrics
2026-11-12T13:39:39Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveID(65074) CurveP256] connIndex=0 e